In [ ]:
from strands import Agent, tool


@tool
def lookup_customer(customer_id: str) -> str:
    """Look up a customer by their ID.

    Args:
        customer_id: The customer ID (e.g. C-1001)
    """
    customer = CUSTOMERS.get(customer_id)
    if not customer:
        return f"No customer found with ID {customer_id}"
    return (
        f"Customer: {customer['name']}\n"
        f"Email: {customer['email']}\n"
        f"Account Status: {customer['account_status']}"
    )

In [ ]:
SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise. Use the available tools to look up customer
information and process requests.

Important guidelines:
- Always verify the customer using lookup_customer before taking action.
- Use tool data to answer questions - don't ask for info you already have.
- Be warm but efficient."""

agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    system_prompt=SYSTEM_PROMPT,
)

result = agent("Hi, I'm customer C-1001. Can you check on my recent orders?")

In [ ]:
import json

print(f"Total messages: {len(agent.messages)}")

for msg in agent.messages:
    role = msg["role"]
    for block in msg.get("content", []):
        if "text" in block:
            print(f"{role}: {block['text'][:100]}")
        elif "toolUse" in block:
            tu = block["toolUse"]
            print(f"tool call: {tu['name']}({json.dumps(tu.get('input', {}))})")

print(f"Cycles: {result.metrics.get_summary()['total_cycles']}")
print(f"Tokens: {result.metrics.accumulated_usage['totalTokens']}")